In [4]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

from notebookutils import mssparkutils

# List the uploaded files
folder_path = "Files/ed_raw"

files = mssparkutils.fs.ls(folder_path)

for file in files:
    print(f"File name: {file.name}")
    print(f"File size: {file.size} bytes")
    print("--------------------------")


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 6, Finished, Available, Finished, False)

File name: ed_hosp-dec23.csv
File size: 43352 bytes
--------------------------
File name: ed_hosp-mar24.csv
File size: 43448 bytes
--------------------------
File name: ed_hosp_dec24.csv
File size: 41283 bytes
--------------------------
File name: ed_hosp_jun24.csv
File size: 41318 bytes
--------------------------
File name: ed_hosp_jun25.csv
File size: 41364 bytes
--------------------------
File name: ed_hosp_mar25.csv
File size: 41240 bytes
--------------------------
File name: ed_hosp_sep24.csv
File size: 43594 bytes
--------------------------
File name: ed_hosp_sep25.csv
File size: 41393 bytes
--------------------------
File name: ed_hospmar23.csv
File size: 41885 bytes
--------------------------
File name: ed_quarterlysep23.csv
File size: 41865 bytes
--------------------------
File name: ed_website_descriptions.xlsx
File size: 13361 bytes
--------------------------
File name: open-data-ed_quarterlyjune23.csv
File size: 41896 bytes
--------------------------


In [5]:

# Read one Emergency Department quarterly CSV
file_path = "Files/ed_raw/ed_hosp-dec23.csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

# Display number of rows and columns
print("Total rows:", df.count())
print("Total columns:", len(df.columns))

# Display column names and data types
df.printSchema()

# Preview first 10 rows
display(df.limit(10))


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 7, Finished, Available, Finished, False)

Total rows: 630
Total columns: 16
root
 |-- Facility/HHS Code: integer (nullable = true)
 |-- Facility/HHS Desc: string (nullable = true)
 |-- Last Month in QTR: string (nullable = true)
 |-- Triage Category: string (nullable = true)
 |-- Number of Attendances: integer (nullable = true)
 |-- Variation in the number of attendances (%): string (nullable = true)
 |-- Patients Seen within clinically recommended times (%): integer (nullable = true)
 |-- Median Waiting time to treatment (minutes): integer (nullable = true)
 |-- Patients who did not wait for treatment (%): string (nullable = true)
 |-- Patients admitted from the Emergency Department (%): string (nullable = true)
 |-- Admissions to hospital within 4 hours (%): string (nullable = true)
 |-- Patients whose ED stay was within 4 hours (%): string (nullable = true)
 |-- Number of patients who left after treatment commenced: integer (nullable = true)
 |-- Percentage of patients who left after treatment commenced (%): string (nullabl

SynapseWidget(Synapse.DataFrame, b19e6b35-fd08-4c88-b702-a47df8de8739)

In [6]:

from pyspark.sql import functions as F

# 1. Identify reporting quarters
print("Reporting periods:")
df.select("Last Month in QTR").distinct().show(
    truncate=False
)

# 2. Identify triage categories
print("Triage categories:")
df.groupBy("Triage Category") \
  .count() \
  .orderBy("Triage Category") \
  .show(truncate=False)

# 3. Count distinct facilities
print("Number of distinct facility codes:")
print(
    df.select("Facility/HHS Code")
      .distinct()
      .count()
)

# 4. Find hospitals associated with Darling Downs
print("Darling Downs facilities:")
df.filter(
    F.lower(F.col("Facility/HHS Desc"))
    .contains("toowoomba")
    |
    F.lower(F.col("Facility/HHS Desc"))
    .contains("darling downs")
).select(
    "Facility/HHS Code",
    "Facility/HHS Desc",
    "Triage Category",
    "Number of Attendances"
).show(30, truncate=False)


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 8, Finished, Available, Finished, False)

Reporting periods:
+-----------------+
|Last Month in QTR|
+-----------------+
|Dec-23           |
+-----------------+

Triage categories:
+---------------+-----+
|Triage Category|count|
+---------------+-----+
|1              |105  |
|2              |105  |
|3              |105  |
|4              |105  |
|5              |105  |
|ALL            |105  |
+---------------+-----+

Number of distinct facility codes:
105
Darling Downs facilities:
+-----------------+------------------+---------------+---------------------+
|Facility/HHS Code|Facility/HHS Desc |Triage Category|Number of Attendances|
+-----------------+------------------+---------------+---------------------+
|104              |Toowoomba Hospital|1              |200                  |
|104              |Toowoomba Hospital|2              |3912                 |
|104              |Toowoomba Hospital|3              |9153                 |
|104              |Toowoomba Hospital|4              |3231                 |
|104            

In [7]:

# Toowoomba Hospital triage performance

df.filter(
    df["Facility/HHS Desc"] == "Toowoomba Hospital"
).select(
    "Triage Category",
    "Number of Attendances",
    "Patients Seen within clinically recommended times (%)",
    "Median Waiting time to treatment (minutes)"
).orderBy("Triage Category").show(truncate=False)


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 9, Finished, Available, Finished, False)

+---------------+---------------------+-----------------------------------------------------+------------------------------------------+
|Triage Category|Number of Attendances|Patients Seen within clinically recommended times (%)|Median Waiting time to treatment (minutes)|
+---------------+---------------------+-----------------------------------------------------+------------------------------------------+
|1              |200                  |100                                                  |0                                         |
|2              |3912                 |57                                                   |9                                         |
|3              |9153                 |55                                                   |27                                        |
|4              |3231                 |72                                                   |32                                        |
|5              |376                  |89

In [8]:

from pyspark.sql import functions as F

df.filter(
    F.col("Facility/HHS Desc") == "Toowoomba Hospital"
).select(
    F.col("Triage Category").alias("Triage"),
    F.col("Number of Attendances").alias("Attendances"),
    F.col("Patients Seen within clinically recommended times (%)").alias("Within_Target_Pct"),
    F.col("Median Waiting time to treatment (minutes)").alias("Median_Wait_Min")
).orderBy("Triage").show(truncate=False)


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 10, Finished, Available, Finished, False)

+------+-----------+-----------------+---------------+
|Triage|Attendances|Within_Target_Pct|Median_Wait_Min|
+------+-----------+-----------------+---------------+
|1     |200        |100              |0              |
|2     |3912       |57               |9              |
|3     |9153       |55               |27             |
|4     |3231       |72               |32             |
|5     |376        |89               |32             |
|ALL   |16872      |60               |21             |
+------+-----------+-----------------+---------------+



In [9]:

from pyspark.sql import functions as F

# Total records
total_rows = df.count()

print(f"Total records: {total_rows}")

# Count null values for every column
missing_counts = df.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

# Display results vertically
missing_df = missing_counts.select(
    F.explode(
        F.map_from_arrays(
            F.array(*[F.lit(c) for c in df.columns]),
            F.array(*[F.col(c) for c in df.columns])
        )
    ).alias("Column", "Missing_Count")
)

display(
    missing_df.orderBy(
        F.desc("Missing_Count")
    )
)


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 11, Finished, Available, Finished, False)

Total records: 630


SynapseWidget(Synapse.DataFrame, 81d381a9-9e17-4160-93d0-8744c2d6800a)

In [10]:

from pyspark.sql import functions as F

# Find records with missing admission timeliness values
missing_admissions = df.filter(
    F.col("Admissions to hospital within 4 hours (%)").isNull()
)

# Show affected facilities and triage categories
display(
    missing_admissions.select(
        F.col("Facility/HHS Code").alias("Facility_Code"),
        F.col("Facility/HHS Desc").alias("Facility"),
        F.col("Triage Category").alias("Triage"),
        F.col("Number of Attendances").alias("Attendances"),
        F.col("Patients admitted from the Emergency Department (%)")
         .alias("Admitted_Pct"),
        F.col("Admissions to hospital within 4 hours (%)")
         .alias("Within_4_Hours_Pct")
    ).orderBy("Facility", "Triage")
)


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c8516538-dc36-4dbd-ba79-5d6aa8dbdf29)

In [11]:

display(
    missing_admissions.groupBy("Triage Category")
    .count()
    .orderBy("Triage Category")
)


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8036cac6-48c3-4494-98cd-cf67677a6ac4)

In [12]:

display(
    missing_admissions.groupBy("Triage Category")
    .agg(
        F.count("*").alias("Missing_Records"),
        F.min("Number of Attendances").alias("Min_Attendances"),
        F.max("Number of Attendances").alias("Max_Attendances"),
        F.avg("Number of Attendances").alias("Avg_Attendances")
    )
    .orderBy("Triage Category")
)


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1c1a17db-4527-4d04-8c2f-03ad3fc2b26b)

In [13]:
from notebookutils import fs
from pyspark.sql import functions as F

# Find all CSV files in the ED raw folder
csv_files = [
    f for f in fs.ls("Files/ed_raw")
    if f.name.lower().endswith(".csv")
]

schema_results = []
column_sets = {}

for file in csv_files:

    # Read columns as strings to compare raw structures
    temp_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(file.path)
    )

    columns = temp_df.columns
    column_sets[file.name] = columns

    schema_results.append({
        "File_Name": file.name,
        "Total_Rows": temp_df.count(),
        "Total_Columns": len(columns),
        "Reporting_Periods": ", ".join(
            str(r[0]) for r in
            temp_df.select("Last Month in QTR")
            .distinct().collect()
        ) if "Last Month in QTR" in columns else "Column missing"
    })

# Show summary
display(
    spark.createDataFrame(schema_results)
)


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6a82404c-a6d3-415e-8a2c-5bd339693aa2)

In [14]:

# Reference: December 2023 dataset
reference_file = "ed_hosp-dec23.csv"

reference_columns = column_sets[reference_file]

print("REFERENCE COLUMNS:")
for i, col in enumerate(reference_columns, 1):
    print(f"{i}. {col}")

print("\n" + "="*60)

# Compare every quarterly file with reference
for filename, columns in sorted(column_sets.items()):

    missing_columns = [
        col for col in reference_columns
        if col not in columns
    ]

    extra_columns = [
        col for col in columns
        if col not in reference_columns
    ]

    print(f"\nFile: {filename}")

    print("Same column order:",
          columns == reference_columns)

    print("Missing columns:", missing_columns)

    print("Extra columns:", extra_columns)


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 16, Finished, Available, Finished, False)

REFERENCE COLUMNS:
1. Facility/HHS Code
2. Facility/HHS Desc
3. Last Month in QTR
4. Triage Category
5. Number of Attendances
6. Variation in the number of attendances (%)
7. Patients Seen within clinically recommended times (%)
8. Median Waiting time to treatment (minutes)
9. Patients who did not wait for treatment (%)
10. Patients admitted from the Emergency Department (%)
11. Admissions to hospital within 4 hours (%)
12. Patients whose ED stay was within 4 hours (%)
13. Number of patients who left after treatment commenced
14. Percentage of patients who left after treatment commenced (%)
15. Number of patients who did not wait for treatment
16. Median Wait Traffic Light


File: ed_hosp-dec23.csv
Same column order: True
Missing columns: []
Extra columns: []

File: ed_hosp-mar24.csv
Same column order: True
Missing columns: []
Extra columns: []

File: ed_hosp_dec24.csv
Same column order: False
Missing columns: ['Triage Category', 'Median Wait Traffic Light']
Extra columns: ['Triage_Cat

In [17]:

from notebookutils import fs
from pyspark.sql import functions as F
from functools import reduce

# Reference schema from December 2023
reference_columns = df.columns

# Exclude September 2022 for initial version
excluded_files = {"open-data-ed_quarterly1sep22.csv"}

csv_files = sorted(
    [
        f for f in fs.ls("Files/ed_raw")
        if f.name.lower().endswith(".csv")
        and f.name not in excluded_files
    ],
    key=lambda f: f.name
)

dataframes = []

for file in csv_files:

    temp_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(file.path)
    )

    # Standardise triage column name
    if "Triage_Category" in temp_df.columns:
        temp_df = temp_df.withColumnRenamed(
            "Triage_Category",
            "Triage Category"
        )

    # Add missing traffic-light column
    if "Median Wait Traffic Light" not in temp_df.columns:
        temp_df = temp_df.withColumn(
            "Median Wait Traffic Light",
            F.lit(None).cast("string")
        )

    # Validate required columns before combining
    missing = [
        col for col in reference_columns
        if col not in temp_df.columns
    ]

    unexpected = [
        col for col in temp_df.columns
        if col not in reference_columns
    ]

    if missing or unexpected:
        raise ValueError(
            f"Unexpected schema in {file.name}: "
            f"missing={missing}, extra={unexpected}"
        )

    # Standardise order and preserve source filename
    temp_df = (
        temp_df.select(*reference_columns)
        .withColumn("source_file", F.lit(file.name))
    )

    dataframes.append(temp_df)

# Combine quarterly datasets by column name
df_combined = reduce(
    lambda a, b: a.unionByName(b),
    dataframes
)

print("Files combined:", len(csv_files))
print("Total records:", df_combined.count())
print("Total columns:", len(df_combined.columns))

display(
    df_combined.groupBy("Last Month in QTR")
    .agg(
        F.count("*").alias("Records"),
        F.countDistinct("Facility/HHS Code").alias("Facilities")
    )
    .orderBy("Last Month in QTR")
)


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 19, Finished, Available, Finished, False)

Files combined: 11
Total records: 6930
Total columns: 17


SynapseWidget(Synapse.DataFrame, ac1b1347-38f8-4100-89e4-192720ea837a)

In [18]:
from pyspark.sql import functions as F

# Find duplicate facility-quarter-triage combinations
duplicates = (
    df_combined
    .groupBy(
        "Facility/HHS Code",
        "Last Month in QTR",
        "Triage Category"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Number of duplicate groups:", duplicates.count())

display(duplicates)


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 20, Finished, Available, Finished, False)

Number of duplicate groups: 0


SynapseWidget(Synapse.DataFrame, d5e74a67-5c0d-4d1f-9da9-9d8ca4ce4a7c)

In [22]:
# Mapping of original column names to Delta-compatible names

column_mapping = {
    "Facility/HHS Code": "facility_code",
    "Facility/HHS Desc": "facility_name",
    "Last Month in QTR": "reporting_quarter",
    "Triage Category": "triage_category",
    "Number of Attendances": "number_of_attendances",
    "Variation in the number of attendances (%)": "attendance_variation_pct",
    "Patients Seen within clinically recommended times (%)": "seen_within_target_pct",
    "Median Waiting time to treatment (minutes)": "median_wait_minutes",
    "Patients who did not wait for treatment (%)": "did_not_wait_pct",
    "Patients admitted from the Emergency Department (%)": "admitted_from_ed_pct",
    "Admissions to hospital within 4 hours (%)": "admissions_within_4h_pct",
    "Patients whose ED stay was within 4 hours (%)": "ed_stay_within_4h_pct",
    "Number of patients who left after treatment commenced": "left_after_treatment_count",
    "Percentage of patients who left after treatment commenced (%)": "left_after_treatment_pct",
    "Number of patients who did not wait for treatment": "did_not_wait_count",
    "Median Wait Traffic Light": "median_wait_traffic_light",
    "source_file": "source_file"
}

# Rename columns without modifying the original DataFrame
df_bronze = df_combined

for old_name, new_name in column_mapping.items():
    df_bronze = df_bronze.withColumnRenamed(
        old_name, new_name
    )

# Check the updated column names
df_bronze.printSchema()


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 24, Finished, Available, Finished, False)

root
 |-- facility_code: string (nullable = true)
 |-- facility_name: string (nullable = true)
 |-- reporting_quarter: string (nullable = true)
 |-- triage_category: string (nullable = true)
 |-- number_of_attendances: string (nullable = true)
 |-- attendance_variation_pct: string (nullable = true)
 |-- seen_within_target_pct: string (nullable = true)
 |-- median_wait_minutes: string (nullable = true)
 |-- did_not_wait_pct: string (nullable = true)
 |-- admitted_from_ed_pct: string (nullable = true)
 |-- admissions_within_4h_pct: string (nullable = true)
 |-- ed_stay_within_4h_pct: string (nullable = true)
 |-- left_after_treatment_count: string (nullable = true)
 |-- left_after_treatment_pct: string (nullable = true)
 |-- did_not_wait_count: string (nullable = true)
 |-- median_wait_traffic_light: string (nullable = true)
 |-- source_file: string (nullable = false)



In [23]:
# Save standardised Bronze data into Fabric Lakehouse

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_ed_quarterly")

print("Bronze Delta table created successfully!")

print("Total records:", df_bronze.count())
print("Total columns:", len(df_bronze.columns))


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 25, Finished, Available, Finished, False)

Bronze Delta table created successfully!
Total records: 6930
Total columns: 17


In [24]:

# Read the saved Bronze table
df_bronze = spark.table("bronze_ed_quarterly")

print("Total records:", df_bronze.count())
print("Total columns:", len(df_bronze.columns))

display(df_bronze.limit(10))


StatementMeta(, 20b82155-0c04-43ff-82e2-359d82e82fad, 26, Finished, Available, Finished, False)

Total records: 6930
Total columns: 17


SynapseWidget(Synapse.DataFrame, 5a913429-7b5e-4085-9a3d-472e22bbd34b)

In [1]:

from pyspark.sql import functions as F

# Check whether the Bronze table exists
table_name = "bronze_ed_quarterly"

if spark.catalog.tableExists(table_name):

    df_bronze = spark.table(table_name)

    print("Bronze table found!")
    print("Total rows:", df_bronze.count())
    print("Total columns:", len(df_bronze.columns))

    df_bronze.printSchema()

else:
    print("Bronze table not found.")
    print("We need to save the harmonised data first.")


StatementMeta(, 8f059c55-34a6-4651-9f2f-386c6333bc51, 3, Finished, Available, Finished, False)

Bronze table found!
Total rows: 6930
Total columns: 17
root
 |-- facility_code: string (nullable = true)
 |-- facility_name: string (nullable = true)
 |-- reporting_quarter: string (nullable = true)
 |-- triage_category: string (nullable = true)
 |-- number_of_attendances: string (nullable = true)
 |-- attendance_variation_pct: string (nullable = true)
 |-- seen_within_target_pct: string (nullable = true)
 |-- median_wait_minutes: string (nullable = true)
 |-- did_not_wait_pct: string (nullable = true)
 |-- admitted_from_ed_pct: string (nullable = true)
 |-- admissions_within_4h_pct: string (nullable = true)
 |-- ed_stay_within_4h_pct: string (nullable = true)
 |-- left_after_treatment_count: string (nullable = true)
 |-- left_after_treatment_pct: string (nullable = true)
 |-- did_not_wait_count: string (nullable = true)
 |-- median_wait_traffic_light: string (nullable = true)
 |-- source_file: string (nullable = true)



In [2]:
# Check duplicate facility-quarter-triage combinations

duplicates = (
    df_bronze
    .groupBy(
        "facility_code",
        "reporting_quarter",
        "triage_category"
    )
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 1)
)

print("Duplicate groups:", duplicates.count())

display(duplicates)


StatementMeta(, 8f059c55-34a6-4651-9f2f-386c6333bc51, 4, Finished, Available, Finished, False)

Duplicate groups: 0


SynapseWidget(Synapse.DataFrame, 81d0d561-aaa4-4316-ae81-6fa23a4b5d38)

In [3]:

# Reporting period completeness

quarter_summary = (
    df_bronze
    .groupBy("reporting_quarter")
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("facility_code").alias("facilities"),
        F.countDistinct("triage_category").alias("triage_categories")
    )
    .orderBy("reporting_quarter")
)

display(quarter_summary)


StatementMeta(, 8f059c55-34a6-4651-9f2f-386c6333bc51, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 841f0710-29c9-438b-aa89-7d588583ac83)